# Perbandingan metode importance
**tabular › explain_classification** · notebook `perm_vs_gain`

**Kapan dipakai:** Mengecek apakah kesimpulan 'fitur penting' stabil antar metode (permutation, gain, SHAP). Kalau berbeda jauh, jangan terlalu percaya satu metode.

**Jebakan umum:** Gain bias ke fitur kardinalitas tinggi. Permutation importance menurun pada fitur berkorelasi (kredit terbagi). Selalu hitung pada data validation, bukan training.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi
Ganti bagian ini saja saat memakai data lomba.

In [ ]:
TRAIN_PATH = ROOT / "data_latihan/tabular/classification/titanic_train.csv"
TEST_PATH  = ROOT / "data_latihan/tabular/classification/titanic_test.csv"      # None kalau tidak ada file test
TARGET     = "survived"
ID_COL     = 'id'
DROP_COLS  = ['alive']
GROUP_COL  = None                      # isi bila ada grup agar tidak bocor
SCORING    = "f1_macro"
SEED, N_SPLITS, VALID_SIZE = 42, 5, 0.2
MODEL_KEY  = "perm_vs_gain"
TASK       = "classification"
KIND       = "tree"
OUT = out_dir("tabular", "explain_classification", MODEL_KEY)
set_seed(SEED)

## 2. Baca data

In [ ]:
df = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH) if TEST_PATH else None
print(df.shape, None if test is None else test.shape)
display(df.head(3))

## 3. Fitur & split

In [ ]:
from superdb.tabular import detect_columns, prepare_features, build_pipeline, split_train_valid, cv_score
cols = detect_columns(df, target=TARGET, id_col=ID_COL, drop_cols=DROP_COLS)
X, NUM, CAT = prepare_features(df, cols)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(df[TARGET])
print("kelas:", dict(enumerate(le.classes_)))
groups = df[GROUP_COL] if GROUP_COL else None
Xtr, Xva, ytr, yva = split_train_valid(X, y, TASK, VALID_SIZE, SEED, groups)
GTR = None if groups is None else groups.iloc[Xtr.index]   # grup untuk CV
print(Xtr.shape, Xva.shape)

## 4. Model yang akan dijelaskan
Penjelasan selalu dihitung pada **validation**, bukan training.

In [ ]:
from lightgbm import LGBMClassifier

model = LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=15, min_child_samples=10,
                       subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=42, n_jobs=-1, verbose=-1)
pipe = build_pipeline(model, NUM, CAT, kind='tree').fit(Xtr, ytr)

In [ ]:
pred = pipe.predict(Xva)
proba = pipe.predict_proba(Xva)
metrics = classification_metrics(yva, pred, proba, classes=np.arange(proba.shape[1]))
display(show_table(metrics))

Matriks fitur setelah preprocessing (nama kolom dipertahankan). `mdl` = model di dalam pipeline.

In [ ]:
from superdb.tabular import feature_names
names = feature_names(pipe)
pre, mdl = pipe.named_steps["pre"], pipe.named_steps["model"]
Ztr = pd.DataFrame(pre.transform(Xtr), columns=names, index=Xtr.index)
Zva = pd.DataFrame(pre.transform(Xva), columns=names, index=Xva.index)
Zva.shape

## 5. Tiga cara mengukur importance

In [ ]:
import shap
from sklearn.inspection import permutation_importance
pi = permutation_importance(pipe, Xva, yva, scoring=SCORING, n_repeats=5, random_state=SEED)
perm = pd.Series(pi.importances_mean, index=Xva.columns)                      # kolom mentah
def raw(n): return n.replace("missingindicator_", "")
gain = pd.Series(mdl.booster_.feature_importance("gain"), index=names)
gain = gain.groupby(raw).sum()
sv = shap.TreeExplainer(mdl)(Zva.sample(min(500, len(Zva)), random_state=SEED))
v = sv.values[..., -1] if sv.values.ndim == 3 else sv.values
shp = pd.Series(np.abs(v).mean(0), index=names).groupby(raw).sum()
cmp = pd.DataFrame({"permutation": perm, "gain": gain, "shap": shp}).fillna(0)
cmp = cmp / cmp.sum()                                                           # normalisasi ke proporsi
display(cmp.sort_values("shap", ascending=False).round(3))

## 6. Seberapa sepakat ketiganya?
Korelasi rank (Spearman) mendekati 1 = kesimpulan stabil.

In [ ]:
display(cmp.corr(method="spearman").round(2))
ax = cmp.sort_values("shap").plot.barh(figsize=(7, max(3, .35 * len(cmp))))
ax.set_title("Importance ternormalisasi: permutation vs gain vs SHAP"); plt.tight_layout()
plt.savefig(OUT / "importance_compare.png", dpi=130); plt.show()

In [ ]:
save_json({"valid": metrics, "method": "perm_vs_gain"}, OUT / "metrics.json")
save_model({"pipe": pipe, "cols": cols}, OUT / "model.joblib"); print("OK. hasil di", OUT)

## Langkah berikutnya
- Penjelasan model adalah bahan bab *analisis* di laporan: tulis 3 temuan utama + satu contoh kasus.
- Fitur yang 'terlalu penting' (mis. satu kolom mendominasi) = curigai kebocoran (lihat `notebooks/tabular/validation/`).